In [ ]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

In [ ]:
!pip install -q -U \
    "protobuf>=5.29,<6.0" \
    "bitsandbytes>=0.45.0" \
    "transformers==4.46.3" \
    "peft==0.13.2" \
    "trl==0.12.1" \
    "accelerate==1.1.1" \
    "datasets==3.1.0" \
    "wandb" \
    "sentencepiece"

In [ ]:
!pip install -q -U "bitsandbytes>=0.45.0"

In [1]:
import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"
os.environ["USE_TORCH"] = "1"

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

In [2]:
import google.protobuf
print(f"Protobuf version: {google.protobuf.__version__}")  # 5.29.x

import transformers
print(f"Transformers: {transformers.__version__}")  # 4.46.3

import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

Protobuf version: 5.29.6
Transformers: 4.46.3
CUDA available: True
GPU: Tesla T4


In [3]:
import bitsandbytes as bnb
print(f"bitsandbytes version: {bnb.__version__}")

import torch
x = torch.randn(10, 10).cuda()
print(f"GPU tensor works: {x.device}")

from bitsandbytes.nn import Linear4bit
layer = Linear4bit(10, 10).cuda()
print("4-bit Linear layer created on GPU — bitsandbytes working")

bitsandbytes version: 0.49.2
GPU tensor works: cuda:0
4-bit Linear layer created on GPU — bitsandbytes working


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

model_id = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=False,
)

print(f"Model loaded. Memory: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded. Memory: 0.71 GB


In [5]:
prompt = "Sarah has 23 apples. She gives 7 to John and buys 12 more. How many apples does Sarah have now?"
messages = [{"role": "user", "content": prompt}]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)

with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=256, do_sample=False)
print(tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True))

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


To determine how many apples Sarah has after her transactions, we can follow these steps:

1. Start with the initial number of apples Sarah has.
   \[
   \text{Initial number of apples} = 23
   \]

2. Subtract the number of apples she gives to John from her initial amount.
   \[
   \text{Apples after giving to John} = 23 - 7 = 16
   \]

3. Add the number of apples she buys to her current total.
   \[
   \text{Final number of apples} = 16 + 12 = 28
   \]

So, after giving 7 apples to John and buying 12 more, Sarah has **28** apples.
